# Ocsai1 Scoring (Trained Models)

This notebook scores Ocsai1 fine-tuned models and writes logprobs CSVs.


Related notebooks:
- `LogProbsGPT4o.ipynb` (vanilla model scoring)
- `LogProbsAnalysis.ipynb` (analysis and figures)


In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
import numpy as np

tqdm.pandas()

data_dir = Path('../../data')

In [33]:
# load original English uses data, from Organisciak et al. 2023
datasets = {}
for split in ['test', 'train']:
    datasets[split] = pd.read_json(data_dir/ 'ocsai1'/ f'finetune-gt_main2_prepared_{split}.jsonl', lines=True)
    def parse_prompt(p):
        parts = p.split('\n')
        prompt = parts[0].split(':', 1)[1]
        response = parts[1].split(':', 1)[1]
        return pd.Series({'prompt': prompt, 'response': response})
    datasets[split][['item', 'response']] = datasets[split]['prompt'].apply(parse_prompt)
    datasets[split]['target'] = pd.to_numeric(datasets[split]['completion']).div(10)
    datasets[split]['type'] = 'uses'
    datasets[split].item.value_counts()

data = datasets['test']
data.item.value_counts()

item
brick         771
box           431
knife         347
rope          295
paperclip     232
bottle        115
book           81
pants          67
tire           66
sock           65
spoon          64
lightbulb      60
ball           57
table          57
fork           56
pencil         54
shoe           53
shovel         52
toothbrush     50
hat            50
backpack        7
Name: count, dtype: int64

# Score

## Score Alternate Uses with Ocsai Models

*wait! Can't use the all data model since it's seen a lot of data, and can't use ocsai 1.5 since it's seen different splits. Need to use the exact ocsai 1 data test/train to ensure fairness*

Two scoring options: legacy completions models (like babbage2 and davinci2), and newer chat models. Legacy models can go up to 5 completions deep, but are more stable at temp=0. Chat models are more variable, but can have log probs up to 20.

Use the legacy first.

In [10]:
# Ocsai-based scoring - for this, you need a custom fine-tuned model on your OpenAI account
# For scoring with a vanilla GPT model, see LogProbsGPT4o.ipynb
from ocsai.inference import Classic_Scorer, Chat_Scorer
from ocsai.prompter import Ocsai1_Chat_Prompter
scorer = Classic_Scorer()
chatscorer = Chat_Scorer(prompter=Ocsai1_Chat_Prompter(),
                             model_dict={'ocsai-chatgpt':'ft:gpt-3.5-turbo-1106:peter-organisciak::8fEKk0V6'})

In [11]:
def score_weighted(row, model='ocsai-babbage2', top_probs=5, scorer=scorer):
    cols = dict()
    try:
        scores = scorer.score(row['item'],
                            response=row['response'], 
                            task_type=row['type'],
                            model=model,
                            top_probs=top_probs,
                            progressive_weighted=True)
        scores = sorted(scores, key=lambda x: x['n'])
        
        for s in scores:
            cols[f'score_{s["n"]}'] = s['score']
            cols[f'confidence_{s["n"]}'] = s['confidence']
    except KeyboardInterrupt:
        raise
    except:
        print(f"Problem with {row['prompt']}")
        for s in range(1, top_probs+1):
            cols[f'score_{s}'] = None
            cols[f'confidence_{s}'] = None
    return pd.Series(cols)

score_weighted(data.iloc[0])

score_1         1.500000
confidence_1    0.223339
score_2         1.252240
confidence_2    0.442711
score_3         1.442894
confidence_3    0.594217
score_4         1.490813
confidence_4    0.730335
score_5         1.463426
confidence_5    0.852724
dtype: float64

In [12]:
score_weighted(data.iloc[0], model='ocsai-chatgpt', top_probs=20, scorer=chatscorer).iloc[::2]

score_1     2.000000
score_2     1.939054
score_3     1.892837
score_4     1.940594
score_5     1.965435
score_6     1.932164
score_7     1.966233
score_8     1.971229
score_9     1.962776
score_10    1.961389
score_11    1.970614
score_12    1.960922
score_13    1.954684
score_14    1.961230
score_15    1.952941
score_16    1.956865
score_17    1.953765
score_18    1.956358
score_19    1.960257
score_20    1.961361
dtype: float64

In [ ]:
#test = data.sample(100)
top_probs = 20
model = 'ocsai-chatgpt'
scorer = chatscorer
responses = data.progress_apply(lambda row: score_weighted(row, model=model, top_probs=top_probs, scorer=scorer), axis=1)
data[responses.columns] = responses.values

In [170]:
data.to_csv(data_dir / 'results/logprobs' / f'logprobs_{model}_ocsai1.csv', index=False)